In [ ]:
import pandas as pd
from mp_api.client import MPRester
import os

# Replace with your actual API key from the Materials Project (obtained from your dashboard or the provided Notion link)
API_KEY = "your_api_key_here"

# Output file path
output_file = "D:/MDS/MDS CBP/data/raw/mp_lakh_compounds.csv"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Batch size for fetching and saving
batch_size = 1000

# Total rows to fetch
total_rows = 100000

# Fields to retrieve (customize as needed for your semiconductor alloy project, e.g., band_gap for semiconductors)
fields = [
    "material_id", "formula_pretty", "elements", "composition",
    "band_gap", "is_metal", "energy_per_atom", "formation_energy_per_atom",
    "magnetic_ordering", "total_magnetization", "theoretical"
]

fetched = 0

with MPRester(API_KEY) as mpr:
    # Optional: Get total count to verify (uncomment if needed)
    # total_available = mpr.materials.summary.count()
    # print(f"Total available materials: {total_available}")
    
    while fetched < total_rows:
        remaining = total_rows - fetched
        current_batch = min(batch_size, remaining)
        
        # Fetch batch (no specific criteria to get a broad dataset; add filters if needed, e.g., band_gap > 0 for semiconductors)
        docs = mpr.materials.summary.search(
            skip=fetched,
            limit=current_batch,
            fields=fields
        )
        
        if not docs:
            print("No more data available.")
            break
        
        # Convert to DataFrame
        batch_df = pd.DataFrame([doc.dict() for doc in docs])
        
        # Save to CSV
        if fetched == 0:
            batch_df.to_csv(output_file, index=False)
        else:
            batch_df.to_csv(output_file, mode='a', header=False, index=False)
        
        fetched += len(docs)
        print(f"Fetched and saved {fetched} rows so far.")

print(f"Dataset collection complete. Saved to {output_file}")